# KLA Image Restoration — Training Notebook**Attach three datasets** (right panel -> *Add Data*):1. `soumikrakshit/div2k-high-resolution-images` — the synthetic-training corpus.2. **`kla-train`** — your `train.zip` uploaded once as a private Kaggle Dataset.3. **`kla-repo`** — this repository, zipped and uploaded as a private Dataset.   The notebook runs the *same* code that ships in the GitHub repo, so there is   only one source of truth.Set **Accelerator = None** for the first run (cells 1-3, no quota), then switchto **GPU P100** and use *Save Version -> Save & Run All* for training.

## Cell 1 — copy our Phase 0/2 code into the notebookUpload the `phase0/` and `phase2/` folders as a small Kaggle Dataset called`kla-code` (a few KB), or paste the files in directly. This cell just putsthem on the import path.

In [ ]:
import os, sys, glob, json, subprocess# Attach this repository as a Kaggle Dataset. We locate it by finding a file# we know is in it, rather than assuming the mount path -- Kaggle names the# mount folder after the dataset slug, which is not always the title you typed.hits = sorted(glob.glob("/kaggle/input/**/src/model/nafnet_sr.py", recursive=True))assert hits, "repo not attached. /kaggle/input contains: " + str(os.listdir("/kaggle/input"))ROOT = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))sys.path.insert(0, os.path.join(ROOT, "src"))print("repo  :", ROOT)CFG   = os.path.join(ROOT, "configs", "degradation_config.json")SPLIT = os.path.join(ROOT, "configs", "split.json")d = sorted(glob.glob("/kaggle/input/**/DIV2K_train_HR", recursive=True))DIV2K = d[0] if d else Noneprint("DIV2K :", DIV2K, "->", len(glob.glob(str(DIV2K) + "/**/*.png", recursive=True)), "photos")KLA = next((os.path.dirname(c) for c in glob.glob("/kaggle/input/**/GT", recursive=True)), None)print("KLA   :", KLA, "->", len(glob.glob(str(KLA) + "/GT/*.npy")), "GT /",      len(glob.glob(str(KLA) + "/NoisyLR/*.npy")), "NoisyLR")assert DIV2K and KLA, "a dataset is missing -- check the Input panel"print("\nall inputs found.")

## Cell 2 — SMOKE TEST (run this first, on Accelerator = None)Twenty seconds, no GPU quota. It checks the model builds, produces exactlydouble-size output, survives odd input sizes and negative pixels, and that atraining step actually produces gradients.**If anything here fails, fix it before touching the GPU.** Burning quota on acrash from a typo is the most avoidable way to lose hours.

In [ ]:
from model.nafnet_sr import smoke_testsmoke_test("cpu")

## Cell 3 — check the synthetic data actually looks like KLA'sThe whole Phase 2 strategy rests on our fake damage matching their realdamage. Look at the pictures and the numbers before spending GPU hours on it.

In [ ]:
import numpy as np, matplotlib.pyplot as pltfrom degrade import Degrader, minmax, to_greyfrom PIL import ImageCFG = os.path.join(CODE, "degradation_config.json")if not os.path.exists(CFG):    CFG = glob.glob("/kaggle/input/**/degradation_config.json", recursive=True)[0]cfg = json.load(open(CFG)); print(json.dumps(cfg, indent=2))D = Degrader(CFG); rng = np.random.default_rng(0)# one synthetic pair from DIV2Kphoto = to_grey(np.asarray(Image.open(sorted(glob.glob(DIV2K + "/**/*.png", recursive=True))[0]), dtype=np.float32) / 255.)crop = photo[:256, :256]gt_s = minmax(crop); lr_s = D(gt_s, rng)# one real pair from KLAgt_r = np.load(sorted(glob.glob(KLA + "/GT/*.npy"))[0])lr_r = np.load(sorted(glob.glob(KLA + "/NoisyLR/*.npy"))[0])fig, ax = plt.subplots(2, 2, figsize=(9, 9))for a, im, t in zip(ax.ravel(), [gt_s, lr_s, gt_r, lr_r],                    ["SYNTHETIC clean", "SYNTHETIC degraded", "KLA clean", "KLA degraded"]):    a.imshow(im, cmap="gray", vmin=0, vmax=1); a.set_title(t); a.axis("off")plt.tight_layout(); plt.show()for name, a in [("synthetic", lr_s), ("KLA real", lr_r)]:    print(f"{name:10s} std {a.std():.3f}  max {a.max():.2f}  min {a.min():+.3f}  %neg {(a<0).mean()*100:.2f}")

## Cell 4 — trainNow switch **Accelerator → GPU P100** and use *Save & Run All (Commit)* so itkeeps running after you close the laptop. Checkpoints land in `/kaggle/working`.Watch the **HARD** column. `easy` will look better; ignore it. The`vs bicubic` figure is the one that says whether the model is doing anythingat all — if it is not clearly positive within a couple of epochs, stop anddebug the data rather than training longer.

In [ ]:
SPLIT = os.path.join(CODE, "split.json")if not os.path.exists(SPLIT):    SPLIT = glob.glob("/kaggle/input/**/split.json", recursive=True)[0]cmd = [    sys.executable, os.path.join(ROOT, "train.py"),    "--photos", DIV2K,    "--real", KLA,    "--cfg", CFG,    "--split", SPLIT,    "--out", "/kaggle/working",    "--width", "32",    "--gt-size", "256",    "--batch", "16",    "--epochs", "30",    "--iters-per-epoch", "800",    "--workers", "2",    "--real-frac", "0.25",]print(" ".join(cmd))subprocess.run(cmd, check=True)

## Cell 5 — plot the training curvesThe gap between easy and hard is the thing to watch. A widening gap meansoverfitting; you would never see it from a random split.

In [ ]:
log = json.load(open("/kaggle/working/training_log.json"))ep = [r["epoch"] for r in log]fig, ax = plt.subplots(1, 2, figsize=(12, 4))ax[0].plot(ep, [r["easy_psnr"] for r in log], label="val_easy")ax[0].plot(ep, [r["hard_psnr"] for r in log], label="val_HARD", lw=2)ax[0].set_title("PSNR (dB)"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)ax[1].plot(ep, [r["easy_ssim"] for r in log], label="val_easy")ax[1].plot(ep, [r["hard_ssim"] for r in log], label="val_HARD", lw=2)ax[1].set_title("SSIM"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)plt.tight_layout(); plt.show()best = max(log, key=lambda r: r["hard_ssim"] * 100 + r["hard_psnr"])print("best epoch on val_hard:", best)